# Running RAG using RAG Blueprint

In this notebook, we will reuse the Roswell case study but time we will run the pipeline using RAG blueprint developed by NVIDA. This blueprint is avaliable at github (https://github.com/NVIDIA-AI-Blueprints/rag/tree/main) and it is a reference implementation of a RAG application pipeline. The pipeline can either be launched on-prem on on the cloud. The goal of the blueprint is give users an implementation of RAG workflow that they can use to develop their own applications on top of without much effort. The blueprint handles all the RAG related components that were manually handled in notebook #3 such as data preparation, chunking, and pipelining.

This blueprint uses the following NIMs:
* LLama 3.1 70B NIM 
* Llama 3.2 Embedding NIM 
* Llama 3.2 Reranking NIM

**Notebook Requirements**:
* Access to a single NVIDIA GPU (A30, A100, H100, L40S, A40) - For the database and related services.
* Access to build.nvidia.com - After the lab, sign up for access to the NVIDIA NGC and build.nvidia.com to access the models used in this lab.
* Docker
* Notebook is designed to be run from host system, and not within docker container.

# Use Case
We will be re-using the Roswell Case Study from notebook #3.

# Environment Setup

**Lab architecture commentary** 

The pipeline of the RAG blueprint:

![RAG Blueprint Diagram](images/arch_diagram.png)

We can run this blueprint either on-prem or on cloud. For this lab, we will be using the following NIMs as shown in the figure above:

* NeMo Retriever Embedding NIM
* NeMo Retriever Reranking NIM
* LLM NIM

Additionally, there is a microservice called "rag-server" that connect the NIMs and database services together in a pipeline. To take a look at these services,look at deploy/compose/docker-compose.yaml. This server is the chain server in the first figure. This connects to all of the NIMs and facillates the flow of data from the GUI component to the back end components. It communicates to the embedding and reranking NIMs and sends the output of that to LLM NIM along with the user's query.

Finally, there is also a serice called "rag-playground". This conists of the front-end GUI. Idea is that this component can be easily replaced with any other applications that uses RAG workflow.

These rag-server service needs to be running for this notebook. We can use cloud endpoints for the NIMs.


## Setup the RAG Blueprint repository.
>> TODO: Do we want to predownload the repository? For now manually adding my ssh public key to launchpad and then downloading this repo. We will need to make changes to deploy/compose/docker-compose.yaml file.




In [ ]:
!git clone git@github.com:NVIDIA-AI-Blueprints/rag.git

* Modify rag/deploy/compose/docker-compose.yaml to connect to the local NIMs. Locate the section in docker-compose.yaml that sets the URLs for the NIMs being used. Overwrite the variables so that they point to local NIM services as shown below:

<img src="images/blueprint_docker-compose.png">
      

* Modify container image name used for nemollm-inference service in rag/deploy/compose/nims.yaml to use 8b model (from 70b). This is to ensure that model is small enough to fit on the local H100 GPU.
* If document parsing results in errors, add parser dependencies to rag/src/Dockerfile. 
    * Line to add after line 15: RUN apt install -y poppler-utils tesseract-ocr libtesseract-dev
    * Line to add after line 42: RUN pip3 install unstructured[all-docs] pi-heif pytesseract

## Launch the Services


In [ ]:
# Setup Environment
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv('./docker/.env', override=True)

print(f"NGC_API_KEY for lab use only     : {os.getenv('NGC_API_KEY')}")
print(f"NVIDIA_API_KEY for lab use only  : {os.getenv('NVIDIA_API_KEY')}")
print(f"Current working dir              : { os.getcwd() }")
print(f"PROJECT_DIR for lab              : {os.getenv('PROJECT_DIR')}")
print(f"LOCAL_NIM_CACHE for lab          : {os.getenv('LOCAL_NIM_CACHE')}")
print(f"locally hosted model for lab     : {os.getenv('NIM_SERVED_MODEL_NAME')}")

!mkdir -p ~/.cache/model-cache
!export MODEL_DIRECTORY=~/.cache/model-cache


In [ ]:
# Stop all prior running containers 
!scripts/stop_containers.sh

In [ ]:
# We will need to pull down containers from NGC
!echo "${NVIDIA_API_KEY}" | docker login nvcr.io -u '$oauthtoken' --password-stdin

# Launch the services and database containers
#!docker compose -f deploy/compose/docker-compose.yaml up -d

# If you made changes to the GUI or any other components, you can use this to rebuild the container:
#!docker compose -f deploy/compose/docker-compose.yaml up -d --build
!export APP_LLM_SERVERURL="nemollm-inference-microservice:8000"
!export APP_EMBEDDINGS_SERVERURL="nemo-retriever-embedding-microservice:8000"
!export APP_RANKING_SERVERURL="nemo-retriever-ranking-microservice:8000"

!USERID=$(id -u) docker compose -f rag/deploy/compose/docker-compose.yaml --profile local-nim up -d --build

# Look at the status of containers
!docker ps


## Ingesting Data Using the RAG Blueprint's API

#### 1. Install Dependencies

In [ ]:
!pip install requests

#### 2. Setup Base Configuration

In [ ]:
import requests
import json
from typing import Dict, Any

IPADDRESS = "0.0.0.0" #Replace this with the correct IP address for the rag-server
RAG_PORT = "8081"
BASE_URL = f"http://{IPADDRESS}:{RAG_PORT}"  # Replace with your rag-server URL

def print_response(response: requests.Response):
    """Helper to print API response."""
    print(f"Status Code: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except json.JSONDecodeError:
        print(response.text)

#### 3. Health Check Endpoint

**Purpose:**
This endpoint performs a health check on the server. It returns a 200 status code if the server is operational.

In [ ]:
# GET /health
url = f"{BASE_URL}/health"
response = requests.get(url)
print_response(response)

#### 4. Upload Document Endpoint

**Purpose:**
This endpoint uploads a document to the vector store. You can specify the collection name where the document should be stored. To speed up the ingestion process, the code is parallelized using `concurrent.futures`.

##### Upload multiple files with `concurrent.futures`

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

def upload_file(file_path: str, collection_name: str):
    url = f"{BASE_URL}/documents"
    files = {"file": open(file_path, "rb")}
    params = {"collection_name": collection_name}
    response = requests.post(url, files=files, params=params)
    print(f"Uploading {os.path.basename(file_path)}...")
    print_response(response)

directory_path = "data/Roswell_Case_Study"  # Replace with your directory path
collection_name = "roswell_case_study" # Collection helps organize the documents for different use cases/projects.

file_paths = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]

with ThreadPoolExecutor() as executor:
    executor.map(lambda file: upload_file(file, collection_name), file_paths)

## Chat with LLM Using RAG Blueprint's API

#### 1. Generate Answer Endpoint

**Purpose:**
This endpoint generates an AI response to a given user message. The system message is specified in the [prompts.yaml](src/prompt.yaml) file. This API retrieves the relevant chunks related to the query from knowledge base, adds them as part of the LLM prompt and returns a streaming response. It supports parameters like temperature, top_p, knowledge base usage, and also generates based on the specified vector collection.

In [ ]:
def print_llm_response(response):
    try:
        # Split the response text into individual JSON objects
        json_objects = response.text.strip().split('\n')
        
        for json_obj in json_objects:
            if json_obj.startswith('data: '):
                json_obj = json_obj[6:]  # Remove 'data: ' prefix
            
            if json_obj:
                try:
                    parsed_json = json.loads(json_obj)
                    if 'choices' in parsed_json and parsed_json['choices']:
                        content = parsed_json['choices'][0]['message']['content']
                        print(content, end='')
                except json.JSONDecodeError:
                    print(f"Failed to parse JSON object: {json_obj}")
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        print(f"Raw response: {response.text}")
        
url = f"{BASE_URL}/generate"
payload = {
    "messages": [
        {"role": "user", "content": "Summarize Statements from Coltman"}
    ],
    "use_knowledge_base": True,
    "temperature": 0.5,
    "top_p": 0.7,
    "max_tokens": 200,
    "top_k": 4,
    "collection_name": "roswell_case_study"
}

response = requests.post(url, json=payload)
print_llm_response(response)